<a href="https://colab.research.google.com/github/jensentke/DS-Work-Folder/blob/main/DSSP_Finetuning_2026_Part_2_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 1. Install compatible Unsloth environment
# ============================================================

!pip install -q -U unsloth

!pip install -q \
  "datasets>=3.4.1,<4.4.0,!=4.0.*,!=4.1.0" \
  "trl>=0.18.2,<=0.24.0,!=0.19.0" \
  "protobuf>=5.29.1,<6.0.0" \
  peft accelerate bitsandbytes sentencepiece huggingface_hub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.85.1 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-storage 2.43.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.9.0 which is incompatible.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
google-cloud-firestore 2.34.1 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-monitoring 2.32.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorboard 2.21.0 requires protobuf<8.0.0,>=6

In [ ]:
%pip install --upgrade \
    "jedi>=0.16,<0.20" \
    "protobuf>=6.33.5,<7" \
    "fsspec[http]==2025.9.0" \
    "gcsfs==2025.9.0"

  Using cached fsspec-2025.9.0-py3-none-any.whl.metadata (10 kB)
Using cached fsspec-2025.9.0-py3-none-any.whl (199 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 67.2 MB/s  0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.36.2
    Uninstalling protobuf-7.36.2:
      Successfully uninstalled protobuf-7.36.2
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.12.0
    Uninstalling fsspec-2025.12.0:
      Successfully uninstalled fsspec-2025.12.0
  Attempting uninstall: gcsfs
    Found existing installation: gcsfs 2025.12.0
    Uninstalling gcsfs-2025.12.0:
      Successfully uninstalled gcsfs-2025.12.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [gcsfs]


In [ ]:
%pip check

No broken requirements found.


In [ ]:
!pip install -q -U unsloth

In [ ]:
# ============================================================
# 2. Check runtime
# ============================================================

import torch
import importlib.metadata as md

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

for pkg in [
    "unsloth", "unsloth_zoo", "transformers", "trl",
    "torch", "datasets", "peft", "bitsandbytes", "protobuf"
]:
    try:
        print(f"{pkg}: {md.version(pkg)}")
    except Exception:
        print(f"{pkg}: not found")

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
unsloth: 2026.11.4
unsloth_zoo: 2026.11.4
transformers: 5.17.0
trl: 1.15.0
torch: 2.14.1+cu130
datasets: 4.8.5
peft: 0.21.2
bitsandbytes: 0.50.2
protobuf: 6.33.6


In [ ]:
# ============================================================
# 3. Mount Google Drive and set dataset paths
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

train_path = "/content/drive/MyDrive/MSDS FineTuning Workshop/ds_6371_qna_500.jsonl"
test_path  = "/content/drive/MyDrive/MSDS FineTuning Workshop/ds_6371_test_qna_whamo.jsonl"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
# 4. Load model and add LoRA adapters
# ============================================================

import torch
import inspect
from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

max_seq_length = 2048
dtype = None
load_in_4bit = True

# Use the 4-bit version if available to reduce memory and download size.
model_name = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=2,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("Model and LoRA adapters are ready.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1112 22:40:45.758000 1615 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1112 22:40:45.839000 1615 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.4: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.11.4 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


Unsloth: compiling each transformer block as one region (LlamaDecoderLayer).
Model and LoRA adapters are ready.


In [ ]:
# ============================================================
# 5. Load raw train/test datasets
# ============================================================

raw_train_ds = load_dataset("json", data_files=train_path, split="train")
raw_test_ds  = load_dataset("json", data_files=test_path, split="train")

print(raw_train_ds)
print(raw_test_ds)

print("\nExample train row:")
print(raw_train_ds[0])

print("\nExample test row:")
print(raw_test_ds[0])

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['conversations'],
    num_rows: 500
})
Dataset({
    features: ['conversations'],
    num_rows: 5
})

Example train row:
{'conversations': [{'role': 'user', 'content': 'What percentage of the grade is homework in DS 6371?'}, {'role': 'assistant', 'content': 'Homework accounts for 10% of the grade in DS 6371.'}]}

Example test row:
{'conversations': [{'role': 'user', 'content': 'How much is the project worth in DS 6371?'}, {'role': 'assistant', 'content': 'The project is worth 10% of the overall grade.'}]}


In [ ]:
# ============================================================
# 6. Normalize dataset into conversational prompt/completion format
# ============================================================

def normalize_role(role):
    role = str(role).lower().strip()

    if role in ["human", "user"]:
        return "user"
    if role in ["gpt", "assistant", "model"]:
        return "assistant"
    if role in ["system"]:
        return "system"

    return role


def normalize_message(msg):
    """
    Convert one message to {'role': ..., 'content': ...}.
    Supports both:
      {'role': 'user', 'content': '...'}
      {'from': 'human', 'value': '...'}
    """
    if "role" in msg and "content" in msg:
        return {
            "role": normalize_role(msg["role"]),
            "content": str(msg["content"])
        }

    if "from" in msg and "value" in msg:
        return {
            "role": normalize_role(msg["from"]),
            "content": str(msg["value"])
        }

    raise ValueError(f"Unknown message format: {msg}")


def row_to_prompt_completion(row):
    """
    Convert one row to:
      {
        'prompt': [{'role': 'user', 'content': question}],
        'completion': [{'role': 'assistant', 'content': answer}]
      }
    """

    # Case 1: already prompt/completion
    if "prompt" in row and "completion" in row:
        prompt = row["prompt"]
        completion = row["completion"]

        if isinstance(prompt, str):
            prompt = [{"role": "user", "content": prompt}]
        if isinstance(completion, str):
            completion = [{"role": "assistant", "content": completion}]

        return {
            "prompt": prompt,
            "completion": completion
        }

    # Case 2: question/answer
    if "question" in row and "answer" in row:
        return {
            "prompt": [
                {
                    "role": "user",
                    "content": str(row["question"])
                }
            ],
            "completion": [
                {
                    "role": "assistant",
                    "content": str(row["answer"])
                }
            ]
        }

    # Case 3: conversations/messages
    conversations_key = None
    if "conversations" in row:
        conversations_key = "conversations"
    elif "messages" in row:
        conversations_key = "messages"

    if conversations_key is not None:
        messages = [normalize_message(m) for m in row[conversations_key]]

        user_message = None
        assistant_message = None

        for msg in messages:
            if msg["role"] == "user" and user_message is None:
                user_message = msg

            elif msg["role"] == "assistant" and user_message is not None:
                assistant_message = msg
                break

        if user_message is None or assistant_message is None:
            raise ValueError(f"Could not find user/assistant pair in row: {row}")

        return {
            "prompt": [
                {
                    "role": "user",
                    "content": user_message["content"]
                }
            ],
            "completion": [
                {
                    "role": "assistant",
                    "content": assistant_message["content"]
                }
            ]
        }

    raise ValueError(f"Unknown row format. Keys found: {list(row.keys())}")


def convert_dataset(raw_ds):
    rows = []

    for row in raw_ds:
        rows.append(row_to_prompt_completion(row))

    return Dataset.from_list(rows)


train_ds = convert_dataset(raw_train_ds)
test_ds = convert_dataset(raw_test_ds)

print(train_ds)
print(test_ds)

print("\nConverted train example:")
print(train_ds[0])

print("\nConverted test example:")
print(test_ds[0])

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 500
})
Dataset({
    features: ['prompt', 'completion'],
    num_rows: 5
})

Converted train example:
{'prompt': [{'content': 'What percentage of the grade is homework in DS 6371?', 'role': 'user'}], 'completion': [{'content': 'Homework accounts for 10% of the grade in DS 6371.', 'role': 'assistant'}]}

Converted test example:
{'prompt': [{'content': 'How much is the project worth in DS 6371?', 'role': 'user'}], 'completion': [{'content': 'The project is worth 10% of the overall grade.', 'role': 'assistant'}]}


In [ ]:
# ============================================================
# 7. Configure SFTTrainer
# ============================================================

training_args = SFTConfig(
    output_dir="outputs_part2",

    # Dataset handling
    max_length=max_seq_length,
    packing=False,
    dataset_num_proc=1,

    # Training hyperparameters
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_steps=60,
    learning_rate=2e-4,
    warmup_steps=5,

    # Optimizer
    optim="adamw_8bit",

    # Logging / reproducibility
    logging_steps=5,
    seed=42,
    report_to="none",

    # Prompt/completion dataset: train on answer/completion side
    completion_only_loss=True,
)

trainer_kwargs = {
    "model": model,
    "train_dataset": train_ds,
    "eval_dataset": test_ds,
    "args": training_args,
}

# TRL newer versions use processing_class; older versions used tokenizer.
if "processing_class" in inspect.signature(SFTTrainer).parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)

print("Trainer configured.")

Unsloth: Tokenizing ["prompt"+"completion"]:   0%|          | 0/500 [00:00<?, ? examples/s]

Unsloth: Building labels for train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["prompt"+"completion"]:   0%|          | 0/5 [00:00<?, ? examples/s]

Unsloth: Building labels for eval dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Trainer configured.


In [ ]:
# ============================================================
# 8. Train
# ============================================================

trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 5,636,096 of 1,241,450,496 (0.45% trained)


Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.


Step,Training Loss
5,2.553234
10,2.139878
15,1.574157
20,1.648482
25,1.482785
30,1.253547
35,1.009467
40,1.094748
45,1.193225
50,0.971063


Unsloth: Restored added_tokens_decoder metadata in outputs_part2/checkpoint-60/tokenizer_config.json.


TrainOutput(global_step=60, training_loss=1.4199324607849122, metrics={'train_runtime': 53.906, 'train_samples_per_second': 8.904, 'train_steps_per_second': 1.113, 'total_flos': 194262819950592.0, 'train_loss': 1.4199324607849122, 'epoch': 0.96})

In [ ]:
# ============================================================
# 9. Inference helper
# ============================================================

import pandas as pd
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

def ask_model(question, max_new_tokens=128):
    messages = [
        {
            "role": "user",
            "content": question
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return response.strip()

In [ ]:
# ============================================================
# 10. Generate predictions on test data
# ============================================================

results = []

for row in test_ds:
    question = row["prompt"][0]["content"]
    expected = row["completion"][0]["content"]
    predicted = ask_model(question)

    results.append({
        "question": question,
        "expected": expected,
        "predicted": predicted
    })

results_df = pd.DataFrame(results)

results_df

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tok

,question,expected,predicted
0,How much is the project worth in DS 6371?,The project is worth 10% of the overall grade.,"The project is worth $1,000,000."
1,What software tools are required for DS 6371?,The course requires SAS OnDemand and RStudio (...,Students are required to use the following sof...
2,What happens if a student misses the oral exam?,Students who do not pass or miss the oral exam...,A student misses the oral exam and is given a ...
3,When is the final exam for DS 6371 scheduled?,"The final exam is on Saturday, April 20, 2024,...",The final exam for DS 6371 is scheduled for No...
4,What does Whamo stand for?,Whamo stands for excitement and respect.,Whamo stands for Whittling and Woodworking.


In [ ]:
# ============================================================
# 11. Install evaluation metrics
# ============================================================

!pip install -q sacrebleu rouge-score bert-score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# ============================================================
# 12. Calculate Exact Match, BLEU, ROUGE-L, BERTScore, and Embedding Cosine
# ============================================================

import numpy as np
import sacrebleu
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F

# ----------------------------
# 1. Exact match
# ----------------------------

def normalize_text(text):
    return " ".join(str(text).lower().strip().split())

def exact_match_score(expected, predicted):
    return normalize_text(expected) == normalize_text(predicted)


# ----------------------------
# 2. BLEU
# ----------------------------

def bleu_score(expected, predicted):
    return sacrebleu.sentence_bleu(predicted, [expected]).score / 100


# ----------------------------
# 3. ROUGE-L F1
# ----------------------------

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

def rouge_l_f1(expected, predicted):
    scores = rouge.score(expected, predicted)
    return scores["rougeL"].fmeasure


# ----------------------------
# 4. BERTScore F1
# ----------------------------

candidate_answers = results_df["predicted"].tolist()
reference_answers = results_df["expected"].tolist()

bert_precision, bert_recall, bert_f1 = bert_score(
    candidate_answers,
    reference_answers,
    lang="en",
    model_type="distilbert-base-uncased",
    device="cpu",
    verbose=True
)


# ----------------------------
# 5. Embedding cosine similarity
# ----------------------------

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

embed_tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)
embed_model = AutoModel.from_pretrained(embedding_model_name)
embed_model.eval()

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

    return torch.sum(token_embeddings * input_mask_expanded, dim=1) / torch.clamp(
        input_mask_expanded.sum(dim=1),
        min=1e-9
    )

def get_sentence_embedding(text):
    encoded_input = embed_tokenizer(
        text,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    with torch.no_grad():
        model_output = embed_model(**encoded_input)

    sentence_embedding = mean_pooling(model_output, encoded_input["attention_mask"])
    sentence_embedding = F.normalize(sentence_embedding, p=2, dim=1)

    return sentence_embedding

def embedding_cosine_similarity(expected, predicted):
    emb_expected = get_sentence_embedding(expected)
    emb_predicted = get_sentence_embedding(predicted)

    return torch.matmul(emb_expected, emb_predicted.T).item()


# ----------------------------
# Build metrics dataframe
# ----------------------------

metric_rows = []

for i, row in results_df.iterrows():
    expected = row["expected"]
    predicted = row["predicted"]

    metric_rows.append({
        "question": row["question"],
        "expected": expected,
        "predicted": predicted,
        "exact_match": exact_match_score(expected, predicted),
        "bleu": bleu_score(expected, predicted),
        "rouge_l_f1": rouge_l_f1(expected, predicted),
        "bertscore_f1": bert_f1[i].item(),
        "embedding_cosine": embedding_cosine_similarity(expected, predicted)
    })

metrics_df = pd.DataFrame(metric_rows)

metrics_df

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

... (12 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,question,expected,predicted,exact_match,bleu,rouge_l_f1,bertscore_f1,embedding_cosine
0,How much is the project worth in DS 6371?,The project is worth 10% of the overall grade.,"The project is worth $1,000,000.",False,0.245495,0.500000,0.837082,0.657663
1,What software tools are required for DS 6371?,The course requires SAS OnDemand and RStudio (...,Students are required to use the following sof...,False,0.027197,0.166667,0.754418,0.485559
2,What happens if a student misses the oral exam?,Students who do not pass or miss the oral exam...,A student misses the oral exam and is given a ...,False,0.060540,0.342857,0.811367,0.709353
3,When is the final exam for DS 6371 scheduled?,"The final exam is on Saturday, April 20, 2024,...",The final exam for DS 6371 is scheduled for No...,False,0.070992,0.266667,0.787330,0.679364
4,What does Whamo stand for?,Whamo stands for excitement and respect.,Whamo stands for Whittling and Woodworking.,False,0.277762,0.666667,0.858939,0.763042
